## Final Project Notebook

In [1]:
# Import Necessary libraries
import nltk
from nltk.corpus import brown
import pickle
import json

# Load both the lemmetizer, the cmu dictionary, and the cmu to ipa conversion table
lemma_dict = pickle.load(open("ant_lemmas.pickle","rb"))
cmu = nltk.corpus.cmudict.dict()
with open("cmu_to_ipa.json", "r") as file:
    cmu_to_ipa = json.load(file)

# Make cmu pronuncations into strings of ipa
cmu_strings = dict()
for word in cmu:
    string_prons = []
    for pron in cmu[word]:
        try:
            string_prons.append(''.join([cmu_to_ipa.get(char) for char in pron]))
        except:
            pass
    cmu_strings[word] = string_prons

# Assemble all the words in the Brown corpus into a dict with the word, pos tag, and frequency
brown_dict = dict()
for pair in brown.tagged_words():
    try:
        brown_dict[pair[0].lower()][2]+=1
        brown_dict[pair[0].lower()][1].add(pair[1])
    except:
        brown_dict[pair[0].lower()] = [pair[0].lower(),{pair[1]},1]

# Seperate out all the words which were tagged as VB or VBD
present_verbs = []
past_verbs = []
for value in brown_dict.values():
    if 'VB' in value[1]:
        present_verbs.append(value[0])
    elif 'VBD' in value[1]:
        past_verbs.append(value[0])

# Lemmatize each verb in the infinitive
lemma_present = []
for verb in present_verbs:
    try:
        lemma_present.append((verb, lemma_dict[verb]))
    except:
        pass

# Lemmatize each verb in the past
lemma_past = []
for verb in past_verbs:
    try:
        lemma_past.append((verb, lemma_dict[verb]))
    except:
        pass

# Match up lemmatized versions of verbs to get our pairs
verb_pairs = []
for verb1 in lemma_present:
    for verb2 in lemma_past:
        if verb1[1]==verb2[1]:
            if cmu_strings.get(verb1[0])==None or cmu_strings.get(verb2[0])==None:
                pass
            else:
                verb_pairs.append((
                    (verb1[0],cmu_strings.get(verb1[0])[0]),
                    (verb2[0],cmu_strings.get(verb2[0])[0])
                ))
            break

# First 10 elements
print(verb_pairs[:10])

[(('place', 'pleɪs'), ('placed', 'pleɪst')), (('charge', 'ʧɑrʤ'), ('charged', 'ʧɑrʤd')), (('praise', 'preɪz'), ('praised', 'preɪzd')), (('term', 'tɝm'), ('termed', 'tɝmd')), (('judge', 'ʤʌʤ'), ('judged', 'ʤʌʤd')), (('investigate', 'ɪnvɛstəgeɪt'), ('investigated', 'ɪnvɛstəgeɪtəd')), (('interest', 'ɪntrəst'), ('interested', 'ɪntrəstəd')), (('number', 'nʌmbɚ'), ('numbered', 'nʌmbɚd')), (('size', 'saɪz'), ('sized', 'saɪzd')), (('act', 'ækt'), ('acted', 'æktəd'))]


### Fasttext

In [5]:
from gensim.models.fasttext import FastText
from gensim.test.utils import common_texts

In [6]:
pronounce_present = [list(verb[0][1]) for verb in verb_pairs]

In [7]:
model = FastText(vector_size=100, window=5, min_count=1, min_n=2, max_n=4)
model.build_vocab(corpus_iterable=pronounce_present)
model.train(corpus_iterable=pronounce_present, total_examples=len(pronounce_present), epochs=10)

(17270, 80000)

In [8]:
vec2 = model.wv["pleɪs"]
vec3 = model.wv['ɪnvɛstəgeɪt']

In [9]:
vec1= model.wv.get_vector("pleɪst")

In [10]:
import numpy as np

print(np.dot(vec1,vec2))
print(np.dot(vec1,vec3))

0.026055861
0.011600741


### GenSim Pretrained

In [11]:
import gensim
from gensim import downloader

In [12]:
glove_vectors50 = gensim.downloader.load('glove-wiki-gigaword-50')

In [13]:
x_list = list()
y_list = list()
for verb in verb_pairs:
    try:
        y_list.append(glove_vectors50.get_vector(verb[1][0]))
        x_list.append(glove_vectors50.get_vector(verb[0][0]))
    except:
        pass
X = np.array(x_list)
y = np.array(y_list)

In [26]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

regr = MLPRegressor(hidden_layer_sizes=(100,100),random_state=1, max_iter=2000, tol=0.1)
regr.fit(X_train, y_train)
regr.score(X_test, y_test)

0.267811119556427

In [18]:
y_pred = regr.predict(X_test)

In [27]:
# See if it can predict a shift to the past tense
right = 0
wrong = 0
for vec in enumerate(y_pred):
    answer = glove_vectors50.similar_by_vector(y_test[vec[0]], topn=1)[0][0]
    if answer in [ele[0] for ele in glove_vectors50.similar_by_vector(vec[1], topn=10)]:
        right+=1
    else:
        wrong+=1

print(right/(right+wrong))

0.17218543046357615


In [16]:
[ele[0] for ele in glove_vectors50.similar_by_vector(X_test[0], topn=10)]

['hate',
 'hatred',
 'shame',
 'racist',
 'anyone',
 'bigotry',
 'racism',
 'afraid',
 'anybody',
 'fear']

In [17]:
glove_vectors50.similar_by_vector(y_pred[4], topn=10)

[('visibly', 0.7569506764411926),
 ('irritated', 0.755580723285675),
 ('shaking', 0.7496856451034546),
 ('trembled', 0.7487367391586304),
 ('pained', 0.7407701015472412),
 ('calmed', 0.7295437455177307),
 ('trembling', 0.7258015871047974),
 ('dejected', 0.7183420062065125),
 ('hoarse', 0.7137667536735535),
 ('rudely', 0.7059953808784485)]

### Simple N-gramming

In [5]:
def n_gram(word,n=3):
    grams = [word[i:i+n] for i in range(len(word)-n+1)]
    return grams

In [20]:
n_grams = set()
for pair in verb_pairs:
    n_grams.update(n_gram(pair[0][1]))
    n_grams.update(n_gram(pair[1][1]))

n_grams_key = list(n_grams)

In [21]:
def make_ngram_vec(word,vec_key,n=3):
    vec = [0]*len(n_grams_key)
    for gram in n_gram(word,n):
        vec[n_grams_key.index(gram)]+=1
    return vec

In [22]:
x_list = list()
y_list = list()
for verb in verb_pairs:
    try:
        y_list.append(make_ngram_vec(verb[1][1],n_grams_key))
        x_list.append(make_ngram_vec(verb[0][1],n_grams_key))
    except:
        pass
X = np.array(x_list)
y = np.array(y_list)

In [23]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

regr = MLPRegressor(random_state=1, max_iter=2000, tol=0.1)
regr.fit(X_train, y_train)


MLPRegressor(max_iter=2000, random_state=1, tol=0.1)

In [24]:
y_pred = regr.predict(X_test)

### Wickelfeature Recreation

In [6]:
cmu_pronuciation = dict()
for word in cmu:
    string_prons = []
    for pron in cmu[word]:
        try:
            string_prons.append([cmu_to_ipa.get(char) for char in pron])
        except:
            pass
    cmu_pronuciation[word] = string_prons

In [11]:
verb_pairs2 = []
for verb1 in lemma_present:
    for verb2 in lemma_past:
        if verb1[1]==verb2[1]:
            if cmu_pronuciation.get(verb1[0])==None or cmu_pronuciation.get(verb2[0])==None:
                pass
            else:
                verb_pairs2.append((
                    (verb1[0],cmu_pronuciation.get(verb1[0])[0]),
                    (verb2[0],cmu_pronuciation.get(verb2[0])[0])
                ))
            break


In [27]:
def wickelphones(word,n=3):
    bounded_word = ['#']+word+['#']
    phones = [bounded_word[i:i+n] for i in range(len(bounded_word)-2)]
    return phones